<a href="https://colab.research.google.com/github/psaw/hse-ai24-dl/blob/stepik/stepik/4.3%20-%20PyTorch_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача классификации (MNIST)

В этом ноутбуке вы обучите полносвязную нейронную сеть для решения задачи классификации на датасете MNIST.

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [4]:
# Check Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

# Define Hyper-parameters
input_size = 784
hidden_size = 500
num_classes = 10
num_epochs = 5
batch_size = 100
learning_rate = 0.001

In [6]:
# MNIST dataset

train_dataset = torchvision.datasets.MNIST(root='../../data',
                                           train=True,
                                           transform=transforms.ToTensor(),
                                           download=True)

test_dataset = torchvision.datasets.MNIST(root='../../data',
                                          train=False,
                                          transform=transforms.ToTensor())

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.43MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 220kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 1.77MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.70MB/s]


In [7]:
# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

In [8]:
# Fully connected neural network
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

model = NeuralNet(input_size, hidden_size, num_classes).to(device)

In [9]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [10]:
# Train the model
total_step = len(train_loader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Move tensors to the configured device
        images = images.reshape(-1, 28*28).to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backprpagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

Epoch [1/5], Step [100/600], Loss: 0.3662
Epoch [1/5], Step [200/600], Loss: 0.2018
Epoch [1/5], Step [300/600], Loss: 0.2983
Epoch [1/5], Step [400/600], Loss: 0.2727
Epoch [1/5], Step [500/600], Loss: 0.1077
Epoch [1/5], Step [600/600], Loss: 0.1554
Epoch [2/5], Step [100/600], Loss: 0.0907
Epoch [2/5], Step [200/600], Loss: 0.0844
Epoch [2/5], Step [300/600], Loss: 0.0640
Epoch [2/5], Step [400/600], Loss: 0.1462
Epoch [2/5], Step [500/600], Loss: 0.0507
Epoch [2/5], Step [600/600], Loss: 0.0950
Epoch [3/5], Step [100/600], Loss: 0.0942
Epoch [3/5], Step [200/600], Loss: 0.0769
Epoch [3/5], Step [300/600], Loss: 0.1138
Epoch [3/5], Step [400/600], Loss: 0.0476
Epoch [3/5], Step [500/600], Loss: 0.1028
Epoch [3/5], Step [600/600], Loss: 0.0924
Epoch [4/5], Step [100/600], Loss: 0.0395
Epoch [4/5], Step [200/600], Loss: 0.0449
Epoch [4/5], Step [300/600], Loss: 0.0366
Epoch [4/5], Step [400/600], Loss: 0.0386
Epoch [4/5], Step [500/600], Loss: 0.0632
Epoch [4/5], Step [600/600], Loss:

In [11]:
# Test the model
# In the test phase, don't need to compute gradients (for memory efficiency)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.reshape(-1, 28*28).to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the test images: {} %'.format(100 * correct / total))

# Save the model checkpoint
torch.save(model.state_dict(), 'model.ckpt')

Accuracy of the network on the test images: 97.89 %
